# Iceberg Lakehouse Analytics

This notebook queries **Apache Iceberg** tables produced by the dbt + DuckDB pipeline.
It connects to the **Nessie REST catalog** running in Docker via **PyIceberg** and demonstrates
schema inspection, snapshot history (time travel), and analytical queries with visualizations.

> **Prerequisites:**
> 1. `make docker-up` — starts both MSSQL and the Nessie catalog server
> 2. `make load-iceberg` — exports dbt tables to Iceberg via Nessie

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from pyiceberg.catalog.rest import RestCatalog

NESSIE_URL = "http://localhost:19120"
WAREHOUSE = str(Path("output/iceberg/warehouse").resolve())
NAMESPACE = "default"

catalog = RestCatalog(
    "lakehouse",
    uri=f"{NESSIE_URL}/iceberg",
    warehouse=WAREHOUSE,
    prefix="main",
)

tables = catalog.list_tables(NAMESPACE)
print(f"Connected to Nessie REST catalog — {len(tables)} tables in '{NAMESPACE}':")
for ns, name in tables:
    print(f"  {ns}.{name}")

## Helper: Load Iceberg Table to DataFrame

A small utility to scan an Iceberg table into a pandas DataFrame.

In [ ]:
def load_table(name: str) -> pd.DataFrame:
    """Load an Iceberg table into a pandas DataFrame."""
    tbl = catalog.load_table(f"{NAMESPACE}.{name}")
    return tbl.scan().to_pandas()

---
## Iceberg Features: Schema Inspection

PyIceberg lets us inspect the schema of each table without reading data.

In [ ]:
TABLE_NAMES = [
    "orders_enriched",
    "rpt_revenue_by_country",
    "rpt_revenue_by_category",
    "rpt_customer_summary",
    "rpt_product_performance",
]

for name in TABLE_NAMES:
    tbl = catalog.load_table(f"{NAMESPACE}.{name}")
    print(f"\n=== {name} ===")
    print(f"  Fields: {len(tbl.schema().fields)}")
    for field in tbl.schema().fields:
        print(f"    {field.name:30s}  {field.field_type}")

## Iceberg Features: Time Travel (Snapshot History)

Every write to an Iceberg table creates an immutable snapshot.
We can list all snapshots to see the history of each table.

In [ ]:
from datetime import datetime, timezone

for name in TABLE_NAMES:
    tbl = catalog.load_table(f"{NAMESPACE}.{name}")
    snapshots = tbl.metadata.snapshots
    print(f"\n=== {name} — {len(snapshots)} snapshot(s) ===")
    for snap in snapshots:
        ts = datetime.fromtimestamp(snap.timestamp_ms / 1000, tz=timezone.utc)
        print(f"  Snapshot {snap.snapshot_id}")
        print(f"    Timestamp : {ts.isoformat()}")
        print(f"    Operation : {snap.summary.operation if snap.summary else 'N/A'}")
        if snap.summary and snap.summary.additional_properties:
            added = snap.summary.additional_properties.get("added-records", "?")
            total = snap.summary.additional_properties.get("total-records", "?")
            print(f"    Records   : {added} added, {total} total")

---
## Data Preview: `orders_enriched`

In [ ]:
df_orders = load_table("orders_enriched")
print(f"{len(df_orders):,} rows")
df_orders.head(15)

---
## Revenue Trends

Monthly revenue aggregated from `rpt_revenue_by_country`.

In [ ]:
df_rev_country = load_table("rpt_revenue_by_country")

df_monthly = (
    df_rev_country
    .groupby(["year", "month"], as_index=False)
    .agg(total_orders=("num_orders", "sum"), total_revenue=("total_revenue", "sum"))
    .sort_values(["year", "month"])
)
df_monthly["period"] = (
    df_monthly["year"].astype(str) + "-" + df_monthly["month"].astype(str).str.zfill(2)
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_monthly["period"], df_monthly["total_revenue"], marker="o", color="teal")
ax.set_xlabel("Period")
ax.set_ylabel("Revenue")
ax.set_title("Monthly Revenue Trend")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df_country_agg = (
    df_rev_country
    .groupby("country", as_index=False)
    .agg(total_revenue=("total_revenue", "sum"), total_orders=("num_orders", "sum"))
    .sort_values("total_revenue", ascending=False)
)

fig = px.bar(
    df_country_agg,
    x="country",
    y="total_revenue",
    color="total_orders",
    title="Total Revenue by Country",
    labels={"total_revenue": "Revenue", "total_orders": "Orders"},
)
fig.show()

---
## Customer Segmentation

Analyse customer spend tiers from `rpt_customer_summary`.

In [ ]:
df_customers = load_table("rpt_customer_summary")
df_customers = df_customers.sort_values("total_spend", ascending=False)
df_customers.head(10)

In [ ]:
# Segment customers by spend tier
bins = [0, 500, 2000, 5000, float("inf")]
labels = ["< 500", "500 - 2k", "2k - 5k", "5k+"]
df_customers["spend_tier"] = pd.cut(df_customers["total_spend"], bins=bins, labels=labels)

tier_counts = df_customers["spend_tier"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart of customer tiers
axes[0].pie(tier_counts.values, labels=tier_counts.index, autopct="%1.0f%%", startangle=90)
axes[0].set_title("Customers by Spend Tier")

# Scatter: orders vs spend
axes[1].scatter(df_customers["total_orders"], df_customers["total_spend"], alpha=0.7, c="teal")
axes[1].set_xlabel("Total Orders")
axes[1].set_ylabel("Total Spend")
axes[1].set_title("Orders vs Spend per Customer")

plt.tight_layout()
plt.show()

---
## Product Performance Analysis

Top products by revenue from `rpt_product_performance`.

In [ ]:
df_products = load_table("rpt_product_performance")
df_products = df_products.sort_values("total_revenue", ascending=False)
df_products

In [ ]:
fig = px.bar(
    df_products,
    x="product_name",
    y="total_revenue",
    color="total_units_sold",
    title="Product Performance — Revenue vs Units Sold",
    labels={"total_revenue": "Revenue", "total_units_sold": "Units Sold"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## Revenue by Category

Category breakdown from `rpt_revenue_by_category`.

In [ ]:
df_category = load_table("rpt_revenue_by_category")
df_category

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df_category["category"], df_category["total_revenue"], color="steelblue")
ax.set_xlabel("Revenue")
ax.set_title("Revenue by Product Category")
plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated:
1. **Nessie REST catalog connection** — multi-engine catalog-as-a-service (DuckDB, Spark, Trino compatible)
2. **Schema inspection** — field names and types for every table
3. **Snapshot history (time travel)** — listing immutable snapshots per table
4. **Analytical queries** — revenue trends, customer segmentation, product performance
5. **Visualizations** — matplotlib and plotly charts

All data is read directly from Iceberg tables (Parquet files under `output/iceberg/warehouse/`),
catalogued via **Apache Nessie** which adds Git-like branching, multi-engine access, and
schema evolution tracking on top of the open table format.